In [ ]:
# ── 1. Setup ─────────────────────────────────────────────────
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from scraper import fetch_website_links, fetch_website_contents

In [2]:
load_dotenv(override=True)
api_key =os.getenv('GEMINI_API_KEY')
gemini=OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=api_key)



In [12]:

# ── 2. System Prompts ─────────────────────────────────────────

# Instructs the model to identify brochure-relevant links and return structured JSON
LINK_SYSTEM_PROMPT="""
You are provided with a list of links found on webpage.
You are able to decide which of the links would be most relevant to include in a brochure about a company,
such as links to an About page, or a Company page, pr Careers/Jobs pages.
You should respond in JSON as in this example:
{
  "links": [
    {"type": "about page", "url": "https://full.url/goes/here/about"},
    {"type": "carrer page", "url": "https://another.full.url/careers"}
  ]
}
"""

# Instructs the model on how to write the final brochure

BROUCHURE_SYSTEM_PROMPT= """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [9]:
# ── 3. Link Selection ─────────────────────────────────────────

def get_links_user_prompt(url):
  user_prompt=f"""
Here is a list of links from the website {url} -
Please decide which of these links are relevant web links for a brochure about a company,
respond with the full https URL in JSON format.get_link_user_prompt.get_link_user_prompt. 
Do not include Terms of Sevice, Privacy, email links.

Links (some might be relative links):
"""
  links=fetch_website_links(url)
  user_prompt+= "\n" .join(links)
  return user_prompt


def select_relevant_links(url):
  response= gemini.chat.completions.create(
    model="gemini-2.0-flash",
    messages=[
      {"role": "system", "content": LINK_SYSTEM_PROMPT},
      {"role": "user", "content": get_links_user_prompt(url)}

    ],
    response_format={"type": 'json_object'}


  )
  links=json.load(response.choices[0].message.content)
  return links
    

In [10]:
# ── 4. Content Aggregation ────────────────────────────────────

def fetch_page_and_all_relevant_links(url):
  content=fetch_website_contents(url)
  relevant_links = select_relevant_links(url)

  result= f" ##Landing Page:\n\n {content}\n\n ###Relevant Links:\n"
  for link in relevant_links:
    result+=f"\n\n#### link: {link["type"]}\n\n"
    result+=fetch_website_contents(link["url"])

  return result


def get_brochure_user_prompt(company_name, url):
   
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]  # Truncate to avoid exceeding token limits
    return user_prompt


In [13]:
# ── 5. Brochure Generation (Streaming) ───────────────────────
def stream_brochure(company_name, url):
  stream=gemini.chat.completions.create(
    model="gemini-2.0-flash",
    messages=[
      {"role": "system", "content": BROUCHURE_SYSTEM_PROMPT},
      {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
    ],
    stream=True
  )
  response=""
  display_handle=display(Markdown(""), display_id=True)

  # Accumulate chunks and re-render the markdown on each update

  for chuck in stream:
    response +=chuck.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)
    
# ── 6. Run It ─────────────────────────────────────────────────
stream_brochure("HuggingFace", "https://huggingface.co")

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 41.101160465s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.goog